# 05 – Ensemble Model & Final Comparison

This notebook:
1. Loads predictions from all base models
2. Trains a Ridge regression meta-learner on the validation set
3. Combines predictions to form the ensemble forecast
4. Computes final evaluation metrics and generates comparison plots

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
from pathlib import Path

from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.models.ensemble import EnsembleModel
from src.visualization.plotter import Plotter
from src.config import RESULTS_DIR, MODELS_SAVED_DIR

plotter = Plotter()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
train = load_processed_data('train')
val   = load_processed_data('val')
test  = load_processed_data('test')

target_col = 'Close'
y_val  = val[target_col].values
y_test = test[target_col].values

In [ ]:
# ── Re-run base models on validation set ────────────────────────────────────
# NOTE: In a production pipeline you would load saved models instead.
# Here we retrain on (train) and predict on val to keep things self-contained.

val_preds  = {}
test_preds = {}

# --- ARIMA ---
try:
    from src.models.arima_model import ARIMAModel
    arima = ARIMAModel(order=(5, 1, 0))
    arima.fit(train[target_col])
    val_preds['arima']  = arima.predict(len(val)).values[:len(val)]
    arima.fit(pd.concat([train[target_col], val[target_col]]))
    test_preds['arima'] = arima.predict(len(test)).values[:len(test)]
    print('ARIMA done.')
except Exception as e:
    print(f'ARIMA skipped: {e}')

# --- XGBoost ---
try:
    from src.models.xgboost_model import XGBoostModel
    feat_cols = [c for c in train.columns if c not in [target_col, 'Open', 'High', 'Low', 'Adj Close']]
    xgb = XGBoostModel()
    xgb.fit(train[feat_cols].fillna(0), train[target_col],
            X_val=val[feat_cols].fillna(0), y_val=val[target_col])
    val_preds['xgboost']  = xgb.predict(val[feat_cols].fillna(0))
    xgb.fit(pd.concat([train, val])[feat_cols].fillna(0),
            pd.concat([train, val])[target_col])
    test_preds['xgboost'] = xgb.predict(test[feat_cols].fillna(0))
    print('XGBoost done.')
except Exception as e:
    print(f'XGBoost skipped: {e}')

# --- LSTM (load saved model if available) ---
lstm_result_path = RESULTS_DIR / 'lstm_predictions.csv'
if lstm_result_path.exists():
    lstm_df = pd.read_csv(lstm_result_path, index_col=0, parse_dates=True)
    test_preds['lstm'] = lstm_df['lstm'].values[:len(test)]
    print('LSTM predictions loaded.')
else:
    print('LSTM predictions not found – run notebook 04 first.')

print(f'Available base models: {list(test_preds.keys())}')

In [ ]:
# ── Train ensemble on validation predictions ─────────────────────────────────
if len(val_preds) < 2:
    print('Need at least 2 base model predictions for ensemble; skipping.')
    ensemble_preds = None
else:
    # Align lengths
    min_val_len  = min(len(v) for v in val_preds.values())
    aligned_val  = {k: v[:min_val_len] for k, v in val_preds.items()}
    y_val_trim   = y_val[:min_val_len]

    # Only include models that have test predictions too
    common_models = [m for m in aligned_val if m in test_preds]
    aligned_val   = {m: aligned_val[m]   for m in common_models}
    aligned_test  = {m: test_preds[m][:len(test)] for m in common_models}

    ensemble = EnsembleModel(base_model_names=common_models)
    ensemble.fit(aligned_val, y_val_trim)
    ensemble_preds = ensemble.predict(aligned_test)

    ensemble_metrics = calculate_metrics(y_test[:len(ensemble_preds)], ensemble_preds)
    print('Ensemble metrics:', ensemble_metrics)
    ensemble.save()
    print('Model weights:', ensemble.get_weights().to_dict())

In [ ]:
# ── Compute per-model metrics ─────────────────────────────────────────────────
all_metrics = {}
all_pred_dict = {}

for model_name, preds in test_preds.items():
    n = min(len(preds), len(y_test))
    all_metrics[model_name] = calculate_metrics(y_test[:n], preds[:n])
    all_pred_dict[model_name] = preds[:n]

if ensemble_preds is not None:
    n = min(len(ensemble_preds), len(y_test))
    all_metrics['ensemble'] = calculate_metrics(y_test[:n], ensemble_preds[:n])
    all_pred_dict['ensemble'] = ensemble_preds[:n]

metrics_df = pd.DataFrame(all_metrics).T
print(metrics_df.to_string())
metrics_df.to_csv(RESULTS_DIR / 'all_models_metrics.csv')

In [ ]:
# ── Visualisations ────────────────────────────────────────────────────────────
plotter.plot_predictions_comparison(
    y_test,
    all_pred_dict,
    dates=test.index,
    title='All Models vs Actual (Test Set)',
    filename='all_models_predictions.png'
)

plotter.plot_metrics_comparison(all_metrics, filename='metrics_comparison.png')
print('All plots saved.')

## Final Summary

| Metric | Best Model |
|--------|------------|
| RMSE   | See `reports/results/all_models_metrics.csv` |
| MAPE   | See `reports/results/all_models_metrics.csv` |
| Dir. Acc. | See `reports/results/all_models_metrics.csv` |

All results saved under `reports/results/`.